In [98]:
import pandas as pd
import numpy as np
from IPython.display import display, HTML

In [99]:
df = pd.read_csv("../data/silver/my1_cleaned.csv", parse_dates=['datetime'], index_col='datetime')
df.head()

,ozone,pm25,vehicle_count,prcp_amt,glbl_irad_amt,air_temperature,rltv_hum,wind_speed,wind_direction,msl_pressure,blh_m
datetime,,,,,,,,,,,
2018-07-17 13:00:00,10.82667,NaN,3449.0,0.0,1598.0,23.1,40.8,9.0,250.0,1015.8,1759.9788
2018-07-17 14:00:00,11.27571,NaN,3445.0,0.0,1403.0,23.4,42.0,8.0,270.0,1015.8,1854.9941
2018-07-17 15:00:00,9.08044,NaN,3496.0,0.0,1168.0,23.0,42.1,10.0,240.0,1015.7,1916.6765
2018-07-17 16:00:00,10.47743,NaN,3628.0,0.0,1001.0,23.0,42.7,9.0,250.0,1015.7,1846.6980
2018-07-17 17:00:00,12.12388,NaN,3675.0,0.0,577.0,22.7,45.1,8.0,250.0,1015.8,1450.4360


## Adding temporal variables

Unix time, julian day, day of week, hour of day

In [100]:
df['unix_time'] = df.index.astype(np.int64) // 10**9
df['julian_day'] = df.index.dayofyear
df['day_of_week'] = df.index.dayofweek
df['hour_of_day'] = df.index.hour

## Transforming variables
Transforming wind speed unit.

In [101]:
# winds speed from knots to m/s
df['wind_speed'] = df['wind_speed'] * 0.514444

#wd_rad = np.deg2rad(df['wind_direction'])
#df['wind_u'] = -df['wind_speed'] * np.sin(wd_rad) # + = east
#df['wind_v'] = -df['wind_speed'] * np.cos(wd_rad) # + = north

# drop
#df.drop(columns=['wind_speed', 'wind_direction'], inplace=True)
df.head()

,ozone,pm25,vehicle_count,prcp_amt,glbl_irad_amt,air_temperature,rltv_hum,wind_speed,wind_direction,msl_pressure,blh_m,unix_time,julian_day,day_of_week,hour_of_day
datetime,,,,,,,,,,,,,,,
2018-07-17 13:00:00,10.82667,NaN,3449.0,0.0,1598.0,23.1,40.8,4.629996,250.0,1015.8,1759.9788,1531832,198,1,13
2018-07-17 14:00:00,11.27571,NaN,3445.0,0.0,1403.0,23.4,42.0,4.115552,270.0,1015.8,1854.9941,1531836,198,1,14
2018-07-17 15:00:00,9.08044,NaN,3496.0,0.0,1168.0,23.0,42.1,5.144440,240.0,1015.7,1916.6765,1531839,198,1,15
2018-07-17 16:00:00,10.47743,NaN,3628.0,0.0,1001.0,23.0,42.7,4.629996,250.0,1015.7,1846.6980,1531843,198,1,16
2018-07-17 17:00:00,12.12388,NaN,3675.0,0.0,577.0,22.7,45.1,4.115552,250.0,1015.8,1450.4360,1531846,198,1,17


# Creating hourly datasets

Hourly datasets with raw variables

In [102]:
# remove index
# drop NaN pollutants
hourly_o3 = df.dropna(subset=['ozone']).reset_index().drop(columns=['datetime', 'pm25'])
hourly_pm25 = df.dropna(subset=['pm25']).reset_index().drop(columns=['datetime', 'ozone'])

display(hourly_o3.head())
display(hourly_pm25.head())

,ozone,vehicle_count,prcp_amt,glbl_irad_amt,air_temperature,rltv_hum,wind_speed,wind_direction,msl_pressure,blh_m,unix_time,julian_day,day_of_week,hour_of_day
0,10.82667,3449.0,0.0,1598.0,23.1,40.8,4.629996,250.0,1015.8,1759.9788,1531832,198,1,13
1,11.27571,3445.0,0.0,1403.0,23.4,42.0,4.115552,270.0,1015.8,1854.9941,1531836,198,1,14
2,9.08044,3496.0,0.0,1168.0,23.0,42.1,5.144440,240.0,1015.7,1916.6765,1531839,198,1,15
3,10.47743,3628.0,0.0,1001.0,23.0,42.7,4.629996,250.0,1015.7,1846.6980,1531843,198,1,16
4,12.12388,3675.0,0.0,577.0,22.7,45.1,4.115552,250.0,1015.8,1450.4360,1531846,198,1,17


,pm25,vehicle_count,prcp_amt,glbl_irad_amt,air_temperature,rltv_hum,wind_speed,wind_direction,msl_pressure,blh_m,unix_time,julian_day,day_of_week,hour_of_day
0,13.2,2402.0,0.0,0.0,20.7,71.4,3.086664,240.0,1014.9,229.4208,1532390,205,1,0
1,22.7,3644.0,0.0,963.0,28.7,42.3,4.629996,240.0,1013.2,1798.7944,1532440,205,1,14
2,17.8,3672.0,0.0,1898.0,28.8,43.6,3.086664,250.0,1013.2,1731.4312,1532444,205,1,15
3,19.8,3507.0,0.0,1304.0,29.3,42.5,4.115552,240.0,1012.9,1655.7957,1532448,205,1,16
4,19.0,3707.0,0.0,1199.0,29.1,40.0,4.115552,250.0,1012.9,1563.8152,1532451,205,1,17


## Creating transformed, daily datasets

Daily means and transformed variables - easier for SR

In [103]:
# removing hour of day, and unix time (we will rebuild that as a linear time varaible)
# removing "categorical" variables (we will rebuild)
daily_df = df.drop(columns=['hour_of_day', 'unix_time', 'julian_day', 'day_of_week'])

print("Size before:", len(daily_df))

# creating means
daily_df['date'] = daily_df.index.date

daily_df = daily_df.reset_index(drop=True)

daily_df = daily_df.groupby('date').mean().reset_index()

print("Size after:", len(daily_df))

# adding a linear time variable | t = 0 for the first day, t = 1 for the last day

# guarantee that the data is sorted by date
daily_df = daily_df.sort_values(by='date').reset_index(drop=True)

daily_df['t'] = np.linspace(0, 1, len(daily_df))

# readd julian day
daily_df['julian_day'] = pd.to_datetime(daily_df['date']).dt.dayofyear

# building jsin and jcos variables: 2pij/365
daily_df['jcos'] = np.cos(2 * np.pi * daily_df['julian_day'] / 365)
daily_df['jsin'] = np.sin(2 * np.pi * daily_df['julian_day'] / 365)

# jsum = jsin + jcos => highs on winter, lows on summer
daily_df['jsum'] = daily_df['jsin'] + daily_df['jcos']

# dropping aux
daily_df.drop(columns=['julian_day', 'jcos', 'jsin'], inplace=True)

# readding day of week
daily_df['day_of_week'] = pd.to_datetime(daily_df['date']).dt.dayofweek

# creating dw_wave: np.cos(1.4 * np.pi * df['dw'] / 14) + np.sin(1.2 * np.pi * df['dw'] / 7)
daily_df['dw_wave'] = np.cos(1.4 * np.pi * daily_df['day_of_week'] / 14) + np.sin(1.2 * np.pi * daily_df['day_of_week'] / 7)

# drop dw
daily_df.drop(columns=['day_of_week'], inplace=True)

daily_df.head()

Size before: 37662
Size after: 1514


,date,ozone,pm25,vehicle_count,prcp_amt,glbl_irad_amt,air_temperature,rltv_hum,wind_speed,wind_direction,msl_pressure,blh_m,t,jsum,dw_wave
0,2018-07-17,12.140512,NaN,3671.583333,0.0,526.916667,21.400000,51.950000,3.986941,246.666667,1016.275000,1194.917698,0.000000,-1.228280,1.463956
1,2018-07-18,15.109113,NaN,3224.807692,0.0,760.846154,20.688462,57.303846,1.919272,248.461538,1018.100000,660.040799,0.000661,-1.240163,1.689613
2,2018-07-19,17.136796,NaN,3083.269231,0.0,801.692308,21.507692,52.526923,1.305896,173.846154,1017.969231,634.820013,0.001322,-1.251680,1.586778
3,2018-07-20,21.995558,NaN,3196.653846,0.0,433.730769,21.007692,57.296154,1.503759,147.692308,1014.211538,472.360625,0.001983,-1.262825,1.143590
4,2018-07-21,43.733977,NaN,3197.615385,0.0,751.500000,22.584615,60.484615,2.097349,247.692308,1013.546154,691.114090,0.002644,-1.273596,0.433884


In [104]:
# building pollutant specific dataframes
# and dropping NaN values for each pollutant
daily_o3 = daily_df.drop(columns=['date', 'pm25']).dropna(subset=['ozone'])
daily_pm25 = daily_df.drop(columns=['date', 'ozone']).dropna(subset=['pm25'])

display(daily_o3.head())
display(daily_pm25.head())

,ozone,vehicle_count,prcp_amt,glbl_irad_amt,air_temperature,rltv_hum,wind_speed,wind_direction,msl_pressure,blh_m,t,jsum,dw_wave
0,12.140512,3671.583333,0.0,526.916667,21.400000,51.950000,3.986941,246.666667,1016.275000,1194.917698,0.000000,-1.228280,1.463956
1,15.109113,3224.807692,0.0,760.846154,20.688462,57.303846,1.919272,248.461538,1018.100000,660.040799,0.000661,-1.240163,1.689613
2,17.136796,3083.269231,0.0,801.692308,21.507692,52.526923,1.305896,173.846154,1017.969231,634.820013,0.001322,-1.251680,1.586778
3,21.995558,3196.653846,0.0,433.730769,21.007692,57.296154,1.503759,147.692308,1014.211538,472.360625,0.001983,-1.262825,1.143590
4,43.733977,3197.615385,0.0,751.500000,22.584615,60.484615,2.097349,247.692308,1013.546154,691.114090,0.002644,-1.273596,0.433884


,pm25,vehicle_count,prcp_amt,glbl_irad_amt,air_temperature,rltv_hum,wind_speed,wind_direction,msl_pressure,blh_m,t,jsum,dw_wave
7,20.550000,3124.692308,0.000000,913.730769,24.069231,59.980769,2.829442,242.692308,1014.134615,742.569693,0.004627,-1.303633,1.463956
8,15.900000,3167.807692,0.000000,1008.461538,24.803846,55.600000,1.939058,164.230769,1015.853846,754.603647,0.005288,-1.312876,1.689613
9,28.023077,3278.730769,0.000000,881.769231,26.934615,55.396154,1.939058,171.538462,1012.800000,743.703681,0.005948,-1.321731,1.586778
10,20.876923,3234.307692,0.476923,789.230769,24.165385,61.326923,2.216066,241.153846,1009.053846,506.096748,0.006609,-1.330194,1.143590
11,11.261538,3243.769231,0.015385,886.923077,19.511538,59.757692,5.441235,217.307692,1006.434615,1288.799575,0.007270,-1.338262,0.433884


# Saving data

Pollutants renamed to "target"

In [105]:
hourly_o3_out = hourly_o3.copy().rename(columns={'ozone': 'target'})
hourly_pm25_out = hourly_pm25.copy().rename(columns={'pm25': 'target'})

daily_o3_out = daily_o3.copy().rename(columns={'ozone': 'target'})
daily_pm25_out = daily_pm25.copy().rename(columns={'pm25': 'target'})

# hourly data
hourly_o3_out.to_csv("../data/gold/my1_hourly_o3.csv", index=False)
hourly_pm25_out.to_csv("../data/gold/my1_hourly_pm25.csv", index=False)

# daily data
daily_o3_out.to_csv("../data/gold/my1_daily_o3.csv", index=False)
daily_pm25_out.to_csv("../data/gold/my1_daily_pm25.csv", index=False)